# Lesson 4.9: AI-Assisted Python

**Analyst:** Meron Welderufael
**Date:** 11/11/2026
**Notebook purpose:** Practice the four AI use patterns and the three-check
verification protocol on a small FinFlow analysis. Build the AI Use Note
habit before starting the Module 4 Project.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv(
    'finflow-enriched-200k.csv',
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)
print(f"Loaded {len(df):,} rows")

Loaded 200,000 rows


## Step 1: Define the calculation.

I will calculate how long each churned customer stayed active before churning by subtracting the signup date from the churn date and converting the result into days using .dt.days.

Then, I will group the churned customers by marketing_name (plan tier) to compare their days active. For each tier, I will calculate the mean, median, 25th percentile (p25), 75th percentile (p75), and customer count.

Expected output: A summary table with one row for each plan tier (marketing_name) and five statistical columns: mean, median, p25, p75, and count.

## Step 2: Prompt AI for the scaffolding.

**prompt**
I have a Pandas DataFrame called df with columns including marketing_name (string), signup_date (datetime), churn_date (datetime), and active (boolean).

For customers where active is False (churned customers), write a Pandas script to calculate how long they were typically active before churning. Also, compare the customer tenure across different plan tiers using marketing_name.

Please include the following:
Calculate the number of days each churned customer was active before churning.
Calculate the mean, median, 25th percentile, 75th percentile, and customer count for each plan tier.
Provide an overall summary of customer tenure.
Explain what each step of the code does in simple terms.

# AI response

### 1. Churned customers with valid dates
churned = df.loc[~df["active"] & df["signup_date"].notna() & df["churn_date"].notna()].copy()

### 2. Tenure in days
churned["tenure_days"] = (churned["churn_date"] - churned["signup_date"]).dt.days

### 3. Remove impossible rows (churn before signup)
churned = churned[churned["tenure_days"] >= 0]

### 4. Overall typical tenure
overall = churned["tenure_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])

### 5. Tenure by tier
by_tier = (
    churned.groupby("marketing_name")["tenure_days"]
           .agg(customers="count", mean="mean", median="median",
                p25=lambda s: s.quantile(0.25),
                p75=lambda s: s.quantile(0.75))
           .round(1)
           .sort_values("median", ascending=False)
)

print(overall, by_tier, sep="\n\n")


## Step 3: Apply the verification protocol.

Filter: ~df["active"] selects customers who have churned. The .notna() checks make sure both signup and churn dates are available before calculating tenure. I used .copy() to avoid Pandas' SettingWithCopyWarning when adding a new column.

Tenure: I subtracted the signup date from the churn date to find how many days each customer stayed before churning. The .dt.days converts the result into whole days. 

Sanity filter: I removed rows where the churn date is before the signup date because those records are not valid and could affect the results.

Overall summary: I used .describe() to get the overall statistics, including the mean, median, count, and percentiles. I would focus more on the median because a few customers with very long tenure can affect the average.

By tier: I grouped customers by marketing name to compare the number of churned customers, average tenure, median tenure, and the 25th and 75th percentiles. I rounded the results to one decimal place and sorted by median tenure from highest to lowest. I also checked the customer count because smaller groups may not give a reliable comparison.

Review and concerns: The code uses standard Pandas methods, and I did not find any deprecated methods or suspicious defaults. One thing I noticed is that the results only include tiers available in the filtered data, so I cannot make conclusions about a tier that is missing. 

## Step 4: Run on small data, then full data.

In [2]:
#Run on small data
df_sample = df.head(500).copy()

churned = df_sample.loc[~df_sample["active"] & df_sample["signup_date"].notna() & df_sample["churn_date"].notna()].copy()

churned["tenure_days"] = (churned["churn_date"] - churned["signup_date"]).dt.days
churned = churned[churned["tenure_days"] >= 0]


overall = churned["tenure_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
by_tier = (
    churned.groupby("marketing_name")["tenure_days"]
           .agg(customers="count", mean="mean", median="median",
                p25=lambda s: s.quantile(0.25),
                p75=lambda s: s.quantile(0.75))
           .round(1)
           .sort_values("median", ascending=False)
)

print(overall, by_tier, sep="\n\n")

count   106.00
mean    237.69
std     145.63
min      15.00
25%     129.00
50%     204.00
75%     332.00
90%     446.50
max     640.00
Name: tenure_days, dtype: float64

                customers   mean  median    p25    p75
marketing_name                                        
Professional           22 267.00  279.50 102.20 418.20
Starter Plan           84 230.00  200.50 138.80 311.20


In [3]:
#Run on the full dataset
churned = df.loc[~df["active"] & df["signup_date"].notna() & df["churn_date"].notna()].copy()

churned["tenure_days"] = (churned["churn_date"] - churned["signup_date"]).dt.days
churned = churned[churned["tenure_days"] >= 0]


overall = churned["tenure_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
by_tier = (
    churned.groupby("marketing_name")["tenure_days"]
           .agg(customers="count", mean="mean", median="median",
                p25=lambda s: s.quantile(0.25),
                p75=lambda s: s.quantile(0.75))
           .round(1)
           .sort_values("median", ascending=False)
)

print(overall, by_tier, sep="\n\n")

count   38713.00
mean      260.10
std       153.90
min         0.00
25%       137.00
50%       238.00
75%       369.00
90%       484.80
max       717.00
Name: tenure_days, dtype: float64

                 customers   mean  median    p25    p75
marketing_name                                         
Enterprise Plus        961 260.10  239.00 144.00 366.00
Professional          8577 259.30  239.00 134.00 368.00
Starter Plan         29175 260.30  238.00 138.00 370.00


## Step 5: Sanity-check.

## Sanity check: expected vs. observed tenure

**Expected baseline**
- The dataset covers approximately two years, so I expect churned customers' tenure to fall roughly between 0 and 730 days.
- Since customers churn at different times, I expect the median tenure to be somewhere between 200 and 300 days.
- Red flags: A median close to 5 days could mean I used the wrong date columns or calculated tenure incorrectly. A median around 50,000 could indicate a date-parsing issue.

**Observed**
- The median tenure from both the initial sample and the full dataset falls within my expected range of 200–300 days, which gives me confidence that the calculation is correct.
- In my first check using only the first 500 rows, the summary table had five columns (customers, mean, median, p25, and p75) but only two plan tiers: Professional and Starter Plan. Enterprise Plus was not represented in that sample.
- After running the analysis on the full dataset, all three plan tiers appeared in the results:

- Enterprise Plus: 961 customers, median tenure of 239 days.
- Professional: 8,577 customers, median tenure of 239 days.
- Starter Plan: 29,175 customers, median tenure of 238 days.
  
- The full dataset shows that all three tiers have very similar median tenure, around 238–239 days. Their mean tenure is also close, ranging from 259.30 to 260.30 days.

**Conclusion**
- My initial results were limited because I was only inspecting the first 500 rows, which did not include Enterprise Plus customers in the churned subset.
- After using the full dataset, all three plan tiers were represented, and the summary table had the expected three rows and five statistical columns.
- Overall, the tenure calculation looks correct, and there is very little difference in typical customer tenure across the three plan tiers.

## Step 6: Practice the debug pattern.

In [5]:
#Run it. You see a KeyError: 'signup_date_str'
df_sample = df.head(500).copy()

churned = df_sample.loc[~df_sample["active"] & df_sample["signup_date"].notna() & df_sample["churn_date"].notna()].copy()

churned["tenure_days"] = (churned["churn_date"] - churned["signup_date_str"]).dt.days
churned = churned[churned["tenure_days"] >= 0]


overall = churned["tenure_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
by_tier = (
    churned.groupby("marketing_name")["tenure_days"]
           .agg(customers="count", mean="mean", median="median",
                p25=lambda s: s.quantile(0.25),
                p75=lambda s: s.quantile(0.75))
           .round(1)
           .sort_values("median", ascending=False)
)

print(overall, by_tier, sep="\n\n")

KeyError: 'signup_date_str'

**Expected AI response:**
What it means: The KeyError: 'signup_date_str' means Pandas is looking for a column named signup_date_str, but that column does not exist in the DataFrame at that moment. The issue is not with Pandas itself; it is most likely a column name mismatch.

Likely cause: My DataFrame already has a signup_date column with datetime values, but I was trying to reference signup_date_str, which I had not created. If I need a string version of the date, I have to create that column first. Otherwise, I should use the existing signup_date column.

In [6]:
#after fix the code
df_sample = df.head(500).copy()

churned = df_sample.loc[~df_sample["active"] & df_sample["signup_date"].notna() & df_sample["churn_date"].notna()].copy()

churned["tenure_days"] = (churned["churn_date"] - churned["signup_date"]).dt.days
churned = churned[churned["tenure_days"] >= 0]


overall = churned["tenure_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
by_tier = (
    churned.groupby("marketing_name")["tenure_days"]
           .agg(customers="count", mean="mean", median="median",
                p25=lambda s: s.quantile(0.25),
                p75=lambda s: s.quantile(0.75))
           .round(1)
           .sort_values("median", ascending=False)
)

print(overall, by_tier, sep="\n\n")

count   106.00
mean    237.69
std     145.63
min      15.00
25%     129.00
50%     204.00
75%     332.00
90%     446.50
max     640.00
Name: tenure_days, dtype: float64

                customers   mean  median    p25    p75
marketing_name                                        
Professional           22 267.00  279.50 102.20 418.20
Starter Plan           84 230.00  200.50 138.80 311.20


## Step 7: Write the AI Use Note.

## AI Use Note

Used Claude throughout this lesson notebook for two patterns:

1. **Scaffolding (Step 2):**  I asked Claude to write a Pandas script to calculate how long churned customers typically remained active before churning and whether the results differed by plan tier. I also asked it to explain each step of the code. The output was correct on the first attempt, including the use of .copy(). The results were consistent with expectations for the tiers represented in the data. However, the Enterprise tier was missing from the comparison. Since it did not appear in the first 500 rows of the approximately 200,000-row dataset I initially inspected, I could not draw a reliable conclusion about Enterprise customers.

2. **Debugging (Step 6):** I shared a KeyError with Claude, along with the surrounding code. Claude correctly identified that I was referencing signup_date_str instead of signup_date. The signup_date_str column had not been created in the DataFrame at that point, while signup_date already existed as a datetime column. This helped me understand the importance of checking column names and data types before referencing them.

Verification Protocol: I followed a read-before-run, small-data-first, and sanity-check-output approach when working with AI-generated code. I reviewed and tested each suggestion rather than accepting it automatically. No AI-generated suggestions were silently incorporated into the analysis. The final notebook reflects my own work, with AI used as a supporting tool for learning, scaffolding, and debugging.
